# Step 13 -- ADP-A Empathy Dataset Preparation

**Phase:** 4 -- Model Training & Adapter Alignment
**Spec authority:** SPEC-400 §4.1, §7.0.1, §10, §11; REQ-400-050-052, REQ-400-CL1, REQ-400-171
**Prerequisites:** Step 12 complete (`finetuning/adp_c_evaluator/adp_c_final/` exists and smoke test passed)
**Output:** `finetuning/adp_a_empathy/data/adp_a_train.jsonl`

---

## Purpose

This notebook prepares the training corpus for **ADP-A**, the Empathy Layer adapter. ADP-A teaches
the base model (Mistral 7B v0.3) conversational warmth and active listening -- specifically:
paraphrasing user emotion, validation without agreement, gentle uncertainty framing, and
non-judgmental tone (REQ-400-051).

## Dataset Mix

| Tier | Dataset | HuggingFace ID | Mix Weight |
|------|---------|---------------|------------|
| 1 | AnnoMI | `AnnoMI` | 30% |
| 1 | Amod counseling conversations | `Amod/mental_health_counseling_conversations` | 25% |
| 2 | ESConv | `thu-coai/esconv` | 20% |
| 3 | MentalChat16K | `ShenLab/MentalChat16K` | 15% |
| 4 | EmpatheticDialogues | `facebook/empathetic_dialogues` | 10% |

## ADP-C Rejection Sampling (REQ-400-171)

All candidate responses are passed through the trained ADP-C adapter from Step 12.
Only `PASS` verdicts are retained. MentalChat16K (synthetic) is excluded entirely if
ADP-C is unavailable -- hard gate per REQ-400-171.

## Forbidden Content (REQ-400-052)

Diagnostic labelling, clinical-authority framing, and directive therapy instructions are
forbidden. The ADP-C filter catches most of these automatically; dataset loaders also apply
length and content pre-filters before ADP-C to reduce inference overhead.

In [ ]:
# -- Cell 0: Setup -----------------------------------------------------------
# Standard library only -- no ML deps needed until the ADP-C filter (Cell 8).
import json
import random
import warnings
from collections import Counter
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
random.seed(42)

# Output directory for the prepared training corpus.
OUTPUT_DIR  = Path("../finetuning/adp_a_empathy/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "adp_a_train.jsonl"

# ADP-C checkpoint from Step 12 -- rejection sampling oracle.
# If this path is missing, MentalChat16K is excluded (REQ-400-171 hard gate).
ADP_C_CHECKPOINT = Path("../finetuning/adp_c_evaluator/adp_c_final")
BASE_MODEL_ID    = "mistralai/Mistral-7B-Instruct-v0.3"

print(f"Output path      : {OUTPUT_PATH.resolve()}")
print(f"ADP-C checkpoint : {ADP_C_CHECKPOINT.resolve()}")
print(f"ADP-C available  : {ADP_C_CHECKPOINT.exists()}")
if not ADP_C_CHECKPOINT.exists():
    print()
    print("WARN: ADP-C checkpoint not found.")
    print("      Cells 1-7 (data loading) can proceed without ADP-C.")
    print("      Cells 8-10 (filtering) require ADP-C -- run Step 12 first.")

## Dataset Schema

Every ADP-A training record uses the same `instruction` / `output` structure as ADP-C,
compatible with Mistral's `[INST]...[/INST]` chat template and SFTTrainer's loss-masking.
The key difference from ADP-C:

| Adapter | `instruction` contains | `output` contains |
|---------|------------------------|-------------------|
| ADP-C | candidate response to audit | JSON verdict (PASS/FAIL/REGENERATE) |
| ADP-A | user emotional message + optional context | target empathetic response |

ADP-A learns to **produce** empathetic responses; ADP-C learns to **evaluate** them.
Both use the same SFT pipeline -- Step 14 mirrors Step 12 structurally.

In [ ]:
# -- Record schema -----------------------------------------------------------

# [CONCEPT] instruction/output SFT format
# At training time, SFTTrainer locates '[/INST] ' in each formatted example
# and sets cross-entropy loss to 0 for all tokens before it. The model only
# trains to predict the 'output' tokens -- it doesn't memorise the instruction.
# This is why the instruction can carry rich context (persona, conversation
# history) without bloating the loss signal.

def make_record(
    user_message: str,
    assistant_response: str,
    source: str,
    conversation_context: str = "",
) -> dict:
    '''
    Build one ADP-A training record.

    instruction : prompt the model sees (persona + optional context + user message)
    output      : target empathetic response the model learns to produce
    meta        : provenance tracking only -- stripped before writing to disk
    '''
    context_block = f"Previous exchange:\n{conversation_context}\n\n" if conversation_context else ""
    instruction = (
        "You are Nikko, a warm and supportive companion. "
        "Respond with empathy and emotional validation. "
        "Do not give diagnoses, directive advice, or claim clinical expertise.\n\n"
        f"{context_block}"
        f"User: {user_message}"
    )
    return {
        "instruction": instruction,
        "output": assistant_response,
        "meta": {"source": source},
    }

print("Schema helper defined. Example record:")
ex = make_record(
    user_message="I have been feeling really low lately and I do not know why.",
    assistant_response=(
        "That sounds really hard, and it makes sense that you want to understand it. "
        "Not knowing the reason can be its own kind of exhausting. "
        "How long have you been feeling this way?"
    ),
    source="example",
)
print(json.dumps({k: v for k, v in ex.items() if k != "meta"}, indent=2))

## Dataset Configurations

**`TARGET_TOTAL`** sets the desired final corpus size after ADP-C filtering.
**`OVERSAMPLE_FACTOR`** compensates for filter rejections: at ~80% pass rate,
2.5x over-sampling reliably hits the target after filtering.

These are the only two parameters to adjust if dataset yield is low.

In [ ]:
# -- Dataset mix configuration (SPEC-400 §7.0.1) ----------------------------

TARGET_TOTAL      = 800   # Desired final corpus size after ADP-C filtering.
OVERSAMPLE_FACTOR = 2.5   # Pre-filter sample multiplier to absorb rejections.

DATASET_MIX = {
    "annomi": {
        "weight":    0.30,
        "hf_name":   "AnnoMI",
        "target_n":  int(TARGET_TOTAL * 0.30 * OVERSAMPLE_FACTOR),
        "license":   "CC BY 4.0",
        "synthetic": False,
        "note": "Expert-annotated motivational interviewing transcripts.",
    },
    "amod": {
        "weight":    0.25,
        "hf_name":   "Amod/mental_health_counseling_conversations",
        "target_n":  int(TARGET_TOTAL * 0.25 * OVERSAMPLE_FACTOR),
        "license":   "CC BY 4.0",
        "synthetic": False,
        "note": "Real counselling Q&A; anonymised professional responses.",
    },
    "esconv": {
        "weight":    0.20,
        "hf_name":   "thu-coai/esconv",
        "target_n":  int(TARGET_TOTAL * 0.20 * OVERSAMPLE_FACTOR),
        "license":   "CC BY-NC 4.0",
        "synthetic": False,
        "note": "Peer-support conversations with labeled strategy tags.",
    },
    "mentalchat": {
        "weight":    0.15,
        "hf_name":   "ShenLab/MentalChat16K",
        "target_n":  int(TARGET_TOTAL * 0.15 * OVERSAMPLE_FACTOR),
        "license":   "CC BY 4.0",
        "synthetic": True,   # MUST pass ADP-C filter (REQ-400-171)
        "note": "Synthetic multi-turn. Hard gate: ADP-C filter required.",
    },
    "empathetic": {
        "weight":    0.10,
        "hf_name":   "facebook/empathetic_dialogues",
        "target_n":  int(TARGET_TOTAL * 0.10 * OVERSAMPLE_FACTOR),
        "license":   "CC BY-NC 4.0",
        "synthetic": False,
        "note": "General empathetic tone filler; lower signal quality.",
    },
}

print(f"Target final corpus size : {TARGET_TOTAL} records")
print(f"Oversample factor        : {OVERSAMPLE_FACTOR}x")
print()
print(f"  {'Dataset':<15}  {'Weight':>8}  {'Pre-filter n':>14}  {'Synthetic':>10}")
print(f"  {'-'*15}  {'-'*8}  {'-'*14}  {'-'*10}")
for name, cfg in DATASET_MIX.items():
    print(f"  {name:<15}  {cfg['weight']:>7.0%}  {cfg['target_n']:>14}  {str(cfg['synthetic']):>10}")

## Section 1 -- AnnoMI

AnnoMI (Thompson et al., 2022) contains 133 expert-annotated motivational interviewing (MI)
sessions. Each utterance is coded for therapist behaviour (reflection, affirmation, open
question, etc.) and client change-talk. Only MI-adherent therapist behaviours are extracted --
those that map directly to REQ-400-051's training objectives.

**Extraction:** consecutive (client_turn -> therapist_turn) pairs where the therapist's coded
behaviour is in the approved set. If the dataset fails to load, a schema is printed for manual
fallback placement at `finetuning/adp_a_empathy/data/annomi_raw.jsonl`.

In [ ]:
# -- AnnoMI loader -----------------------------------------------------------
from datasets import load_dataset

# MI-adherent behaviours that align with REQ-400-051.
# Excluded: giving_information (directive), confrontation (REQ-400-052 violation).
ANNOMI_ALLOWED = {
    "reflection",
    "affirmation",
    "emphasising_autonomy",
    "seeking_collaboration",
    "open_question",
}

def load_annomi(target_n: int) -> list:
    '''Load AnnoMI; extract MI-adherent (client -> therapist) turn pairs.'''
    try:
        ds = load_dataset("AnnoMI", split="train", trust_remote_code=True)
    except Exception as e:
        print(f"  AnnoMI load failed: {e}")
        print("  Manual fallback: place JSONL at finetuning/adp_a_empathy/data/annomi_raw.jsonl")
        print("  Required fields: utterance_text (str), interlocutor ('client'/'therapist'),")
        print("                   main_therapist_behaviour (str), session_id (str/int)")
        return []

    # Group by session and sort by utterance order.
    sessions = {}
    for row in ds:
        sid = str(row.get("session_id") or row.get("video_id") or "unknown")
        sessions.setdefault(sid, []).append(row)
    for sid in sessions:
        sessions[sid].sort(key=lambda r: r.get("timestamp") or r.get("utterance_id") or 0)

    records = []
    for sid, utterances in sessions.items():
        for i in range(1, len(utterances)):
            prev = utterances[i - 1]
            curr = utterances[i]
            behaviour = (curr.get("main_therapist_behaviour") or "").lower().replace(" ", "_")
            if (
                prev.get("interlocutor") == "client"
                and curr.get("interlocutor") == "therapist"
                and behaviour in ANNOMI_ALLOWED
            ):
                user_msg  = (prev.get("utterance_text") or "").strip()
                therapist = (curr.get("utterance_text") or "").strip()
                if len(user_msg) > 20 and len(therapist) > 20:
                    records.append(make_record(user_msg, therapist, source="annomi"))

    random.shuffle(records)
    sampled = records[:target_n]
    print(f"  AnnoMI: {len(records)} valid pairs, {len(sampled)} sampled (target {target_n})")
    return sampled

annomi_records = load_annomi(DATASET_MIX["annomi"]["target_n"])

## Section 2 -- Amod / mental_health_counseling_conversations

Counselling Q&A pairs from licensed mental health professionals, scraped and anonymised.
Schema is simple: `Context` (user's situation) and `Response` (counselor's answer).

Length filter: 30-600 characters on the response. Below 30 is too sparse; above 600 tends
toward lecture-style responses that violate Comfort Mode's brevity norm.

In [ ]:
# -- Amod loader -------------------------------------------------------------

def load_amod(target_n: int) -> list:
    '''Load Amod mental_health_counseling_conversations. Schema: Context, Response.'''
    try:
        ds = load_dataset(
            "Amod/mental_health_counseling_conversations",
            split="train",
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"  Amod load failed: {e}")
        print("  Manual fallback: place JSONL at finetuning/adp_a_empathy/data/amod_raw.jsonl")
        print("  Required fields: Context (str), Response (str)")
        return []

    records = []
    for row in ds:
        user_msg = (row.get("Context") or row.get("context") or "").strip()
        response = (row.get("Response") or row.get("response") or "").strip()
        if len(user_msg) > 15 and 30 <= len(response) <= 600:
            records.append(make_record(user_msg, response, source="amod"))

    random.shuffle(records)
    sampled = records[:target_n]
    print(f"  Amod: {len(records)} valid pairs, {len(sampled)} sampled (target {target_n})")
    return sampled

amod_records = load_amod(DATASET_MIX["amod"]["target_n"])

## Section 3 -- ESConv (Emotional Support Conversation)

Multi-turn peer-support dataset where a seeker describes an emotional problem and a supporter
responds with labeled strategies. Approved strategies map to SPEC-400 §4.1 objectives:

- `Restatement or Paraphrasing` -> paraphrasing user emotion (REQ-400-051)
- `Reflection of Feelings` -> emotional validation
- `Affirmation and Reassurance` -> validation without agreement
- `Providing Suggestions` -> included; ADP-C filter catches directive phrasing

Two prior turns of context are included in the instruction, teaching ADP-A to track
conversational continuity rather than just respond to isolated single messages.

In [ ]:
# -- ESConv loader -----------------------------------------------------------

ESCONV_ALLOWED = {
    "Restatement or Paraphrasing",
    "Reflection of Feelings",
    "Affirmation and Reassurance",
    "Providing Suggestions",
    "Self-disclosure",
}

def load_esconv(target_n: int) -> list:
    '''Load ESConv. Schema: list of conversations with a 'dialog' key.'''
    try:
        ds = load_dataset("thu-coai/esconv", split="train", trust_remote_code=True)
    except Exception as e:
        print(f"  ESConv load failed: {e}")
        print("  Manual fallback: place JSONL at finetuning/adp_a_empathy/data/esconv_raw.jsonl")
        print('  Required fields: dialog (list of {speaker:"usr"|"sys", text:str, strategy:str})')
        return []

    records = []
    for conv in ds:
        dialog = conv.get("dialog", [])
        if isinstance(dialog, str):
            try:
                dialog = json.loads(dialog)
            except json.JSONDecodeError:
                continue

        for i in range(1, len(dialog)):
            prev = dialog[i - 1]
            curr = dialog[i]
            if (
                prev.get("speaker") == "usr"
                and curr.get("speaker") == "sys"
                and curr.get("strategy", "") in ESCONV_ALLOWED
            ):
                user_msg = (prev.get("text") or "").strip()
                response = (curr.get("text") or "").strip()
                # Include up to 2 prior turns as context for conversational continuity.
                context_turns = []
                for j in range(max(0, i - 3), i - 1):
                    spkr = "User" if dialog[j].get("speaker") == "usr" else "Nikko"
                    context_turns.append(f"{spkr}: {(dialog[j].get('text') or '').strip()}")
                context = "\n".join(context_turns)
                if len(user_msg) > 15 and len(response) > 20:
                    records.append(make_record(user_msg, response, "esconv", context))

    random.shuffle(records)
    sampled = records[:target_n]
    print(f"  ESConv: {len(records)} valid pairs, {len(sampled)} sampled (target {target_n})")
    return sampled

esconv_records = load_esconv(DATASET_MIX["esconv"]["target_n"])

## Section 4 -- MentalChat16K

MentalChat16K is **synthetic** -- generated by an LLM, not from real counselling sessions.
Per REQ-400-171, synthetic data MUST be reviewed and filtered by the Evaluator (ADP-C) before
inclusion. This is enforced in Cell 9: if ADP-C is unavailable, MentalChat16K contributes
zero records to the final dataset regardless of this loader's output.

It contributes the lowest-weighted 15% for tonal variety, not signal quality.

In [ ]:
# -- MentalChat16K loader ----------------------------------------------------

def load_mentalchat(target_n: int) -> list:
    '''Load MentalChat16K synthetic dataset. ADP-C filter is mandatory (REQ-400-171).'''
    try:
        ds = load_dataset("ShenLab/MentalChat16K", split="train", trust_remote_code=True)
    except Exception as e:
        print(f"  MentalChat16K load failed: {e}")
        print("  Manual fallback: place JSONL at finetuning/adp_a_empathy/data/mentalchat_raw.jsonl")
        print("  Required fields: input (str), output (str)  [or: question/answer]")
        return []

    records = []
    for row in ds:
        # Handle two column naming conventions seen across HuggingFace versions.
        user_msg = (row.get("input") or row.get("question") or row.get("user") or "").strip()
        response = (row.get("output") or row.get("answer") or row.get("assistant") or "").strip()
        if len(user_msg) > 15 and 30 <= len(response) <= 700:
            records.append(make_record(user_msg, response, source="mentalchat"))

    random.shuffle(records)
    sampled = records[:target_n]
    print(f"  MentalChat16K: {len(records)} valid pairs, {len(sampled)} sampled (target {target_n})")
    print("  NOTE: REQ-400-171 hard gate -- ADP-C filter required before final inclusion.")
    return sampled

mentalchat_records = load_mentalchat(DATASET_MIX["mentalchat"]["target_n"])

## Section 5 -- EmpatheticDialogues

EmpatheticDialogues (Facebook Research) contains 25k conversations where one person describes
an emotional situation and another responds empathetically. Lower signal quality than
MI-annotated corpora, included for tonal variety across a broad emotion space.

Only the first response per conversation (`utterance_idx == 0`) is extracted -- the initial
empathetic acknowledgement is the highest-value turn for ADP-A. The dataset encodes literal
commas as `_comma_` (CSV artifact); these are normalised on load.

In [ ]:
# -- EmpatheticDialogues loader ----------------------------------------------

def load_empathetic(target_n: int) -> list:
    '''Load EmpatheticDialogues. Schema: prompt (str), utterance (str), utterance_idx (int).'''
    try:
        ds = load_dataset(
            "facebook/empathetic_dialogues", split="train", trust_remote_code=True
        )
    except Exception as e:
        print(f"  EmpatheticDialogues load failed: {e}")
        print("  Manual fallback: place JSONL at finetuning/adp_a_empathy/data/empathetic_raw.jsonl")
        print("  Required fields: prompt (str), utterance (str), utterance_idx (int)")
        return []

    records = []
    for row in ds:
        if row.get("utterance_idx", 1) != 0:
            continue   # Only the first response per conversation.
        user_msg = (row.get("prompt") or "").strip().replace("_comma_", ",")
        response = (row.get("utterance") or "").strip().replace("_comma_", ",")
        if len(user_msg) > 15 and len(response) > 20:
            records.append(make_record(user_msg, response, source="empathetic_dialogues"))

    random.shuffle(records)
    sampled = records[:target_n]
    print(f"  EmpatheticDialogues: {len(records)} valid pairs, {len(sampled)} sampled (target {target_n})")
    return sampled

empathetic_records = load_empathetic(DATASET_MIX["empathetic"]["target_n"])

In [ ]:
# -- Pre-filter summary ------------------------------------------------------
all_raw = {
    "annomi":     annomi_records,
    "amod":       amod_records,
    "esconv":     esconv_records,
    "mentalchat": mentalchat_records,
    "empathetic": empathetic_records,
}
total_raw = sum(len(v) for v in all_raw.values())

print("Pre-filter dataset summary:")
print(f"  {'Dataset':<20}  {'Loaded':>8}  {'Target':>8}  {'Status'}")
print(f"  {'-'*20}  {'-'*8}  {'-'*8}  {'-'*22}")
for name, recs in all_raw.items():
    target = DATASET_MIX[name]["target_n"]
    status = "OK" if len(recs) >= int(target * 0.5) else "LOW YIELD -- check loader"
    print(f"  {name:<20}  {len(recs):>8}  {target:>8}  {status}")
print(f"  {'TOTAL':<20}  {total_raw:>8}")

## Section 6 -- ADP-C Rejection Sampling Filter

Each candidate response is passed through the trained ADP-C adapter. Only `PASS` verdicts
are retained. This section loads Mistral 7B + ADP-C LoRA in 4-bit (~4 GB VRAM).

**Runtime:** ~1.5 records/second on RTX 3070. At 2000 pre-filter records, expect ~22 minutes.
Progress is logged every 50 records.

**Skip condition:** if `adp_c_final/` is missing, non-synthetic datasets pass unfiltered with
a warning; MentalChat16K is excluded entirely (REQ-400-171 hard gate).

> **Important note on `load_best_model_at_end`:** Step 12 configured `load_best_model_at_end=True`,
> which reloads the checkpoint with the lowest eval loss before saving to `adp_c_final/`. With
> only 7 eval examples, eval loss is noisy -- the checkpoint selected as "best" may be an
> early-epoch checkpoint, not the most converged one. If filter verdicts look wrong (e.g.,
> suspiciously high rejection rate on clean data), test manually against a later checkpoint
> from `finetuning/adp_c_evaluator/checkpoints/` by pointing `ADP_C_CHECKPOINT` at it.

In [ ]:
# -- ADP-C filter setup -----------------------------------------------------
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"

# On Windows: setting PYTORCH_CUDA_ALLOC_CONF before any torch import prevents
# WDDM memory fragmentation. Already set here before torch is imported.

ADPC_AVAILABLE = ADP_C_CHECKPOINT.exists()

if ADPC_AVAILABLE:
    import gc
    import re
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    SYSTEM_PROMPT_ADPC = (
        "You are the NIKKO Evaluator. Audit the response for safety violations, "
        "tone compliance, and hallucination. "
        "Output a JSON object with keys: verdict, triggered_rule, rationale."
    )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    print("Loading tokenizer from ADP-C checkpoint...")
    adpc_tok = AutoTokenizer.from_pretrained(str(ADP_C_CHECKPOINT))
    if adpc_tok.pad_token is None:
        adpc_tok.pad_token = adpc_tok.eos_token

    print("Loading base model (uses HuggingFace cache from Step 12)...")

    # [CONCEPT] PeftModel.from_pretrained for inference
    # Unlike training (where get_peft_model injects new adapter matrices),
    # PeftModel.from_pretrained loads saved A and B matrices from disk and
    # overlays them on the frozen base. Base weights are never modified.
    adpc_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory={0: "5000MiB", "cpu": "16GiB"},
        low_cpu_mem_usage=True,
    )
    adpc_base.config.use_cache = True
    adpc_model = PeftModel.from_pretrained(adpc_base, str(ADP_C_CHECKPOINT))
    adpc_model.eval()

    vram_gb = torch.cuda.memory_allocated(0) / 1024**3 if torch.cuda.is_available() else 0
    print(f"ADP-C ready. VRAM in use: {vram_gb:.2f} GB")
else:
    print("WARN: ADP-C checkpoint not found.")
    print("      Non-synthetic datasets will pass unfiltered.")
    print("      MentalChat16K EXCLUDED (REQ-400-171 hard gate).")

In [ ]:
# -- ADP-C filter function ---------------------------------------------------

def adpc_score(response: str) -> str:
    '''
    Run one response through ADP-C; return 'PASS', 'FAIL', 'REGENERATE', or 'ERROR'.
    Truncated at 512 tokens to match Step 12 training seq length.
    '''
    prompt = (
        f"<s>[INST] {SYSTEM_PROMPT_ADPC}\n\n"
        "Operational mode: COMFORT\n"
        "Evidence context: No evidence provided.\n\n"
        f"RESPONSE TO AUDIT:\n{response} [/INST] "
    )
    inputs = adpc_tok(prompt, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        out = adpc_model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.05,
            do_sample=True,
            pad_token_id=adpc_tok.eos_token_id,
        )
    decoded = adpc_tok.decode(
        out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    ).strip()
    m = re.search(r"\{.*\}", decoded, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0)).get("verdict", "ERROR")
        except json.JSONDecodeError:
            pass
    return "ERROR"


def filter_dataset(records: list, name: str, is_synthetic: bool = False) -> list:
    '''
    Filter records through ADP-C. Returns only PASS verdicts.
    If ADP-C unavailable: empty list for synthetic data; unfiltered for others.
    '''
    if not ADPC_AVAILABLE:
        if is_synthetic:
            print(f"  [{name}] EXCLUDED -- synthetic data requires ADP-C (REQ-400-171).")
            return []
        print(f"  [{name}] WARNING -- ADP-C unavailable. {len(records)} records pass unfiltered.")
        return records

    passed, rejected = [], []
    for i, record in enumerate(records):
        if i % 50 == 0 and i > 0:
            rate = len(passed) / i * 100
            print(f"  [{name}] {i}/{len(records)}  pass={len(passed)} ({rate:.0f}%)...")
        verdict = adpc_score(record["output"])
        if verdict == "PASS":
            passed.append(record)
        else:
            rejected.append(record)

    rate = len(passed) / len(records) * 100 if records else 0
    print(f"  [{name}] Done. Pass: {len(passed)} ({rate:.1f}%)  Reject: {len(rejected)}")
    return passed


print("Filter function defined. Running filter across all datasets...")
print("(15-30 minutes depending on pre-filter corpus size.)")
print()
annomi_filtered     = filter_dataset(annomi_records,     "annomi",     is_synthetic=False)
amod_filtered       = filter_dataset(amod_records,       "amod",       is_synthetic=False)
esconv_filtered     = filter_dataset(esconv_records,     "esconv",     is_synthetic=False)
mentalchat_filtered = filter_dataset(mentalchat_records, "mentalchat", is_synthetic=True)
empathetic_filtered = filter_dataset(empathetic_records, "empathetic", is_synthetic=False)
print()
print("Filter complete.")

## Section 7 -- Assemble Final Dataset

Each filtered dataset is sampled to its target contribution (`weight x TARGET_TOTAL`).
If a dataset's filtered yield is below target, a warning is raised. Options:
1. Increase `OVERSAMPLE_FACTOR` in Cell 2 and rerun Cells 3-9
2. Accept the reduced corpus size (minimum viable for Step 14: 600 records)

In [ ]:
# -- Apply mix ratios and write to disk -------------------------------------

filtered_sets = {
    "annomi":     annomi_filtered,
    "amod":       amod_filtered,
    "esconv":     esconv_filtered,
    "mentalchat": mentalchat_filtered,
    "empathetic": empathetic_filtered,
}

final_records = []
print(f"Assembling final dataset (target: {TARGET_TOTAL})...")
print(f"  {'Dataset':<20}  {'Target':>8}  {'Available':>10}  {'Sampled':>10}  Status")
print(f"  {'-'*20}  {'-'*8}  {'-'*10}  {'-'*10}  ------")

for name, records in filtered_sets.items():
    weight   = DATASET_MIX[name]["weight"]
    target_n = int(TARGET_TOTAL * weight)
    sampled  = records[:target_n]
    status   = "OK" if len(records) >= target_n else f"SHORT by {target_n - len(records)}"
    print(f"  {name:<20}  {target_n:>8}  {len(records):>10}  {len(sampled):>10}  {status}")
    final_records.extend(sampled)

# Strip meta field -- provenance labels are not seen by the model during training.
for r in final_records:
    r.pop("meta", None)

random.shuffle(final_records)

print(f"  {'TOTAL':<20}  {TARGET_TOTAL:>8}  {'':>10}  {len(final_records):>10}")
print()

# -- Write to disk -----------------------------------------------------------
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for record in final_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

file_kb = OUTPUT_PATH.stat().st_size / 1024
print(f"Dataset written  : {OUTPUT_PATH.resolve()}")
print(f"Total records    : {len(final_records)}")
print(f"File size        : {file_kb:.1f} KB")

# -- Integrity check ---------------------------------------------------------
loaded      = [json.loads(l) for l in OUTPUT_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
n_miss_inst = sum(1 for r in loaded if not r.get("instruction", "").strip())
n_miss_out  = sum(1 for r in loaded if not r.get("output", "").strip())
print(f"Missing instruction: {n_miss_inst}")
print(f"Missing output     : {n_miss_out}")
if n_miss_inst == 0 and n_miss_out == 0:
    print()
    print("All records valid. adp_a_train.jsonl is ready for Step 14.")
else:
    print("WARN: malformed records found. Review the affected dataset loaders.")

## Summary and Next Steps

This notebook produced `finetuning/adp_a_empathy/data/adp_a_train.jsonl` -- the filtered,
mix-weighted training corpus for ADP-A. Each record contains:
- `instruction`: Nikko persona prompt + optional conversation context + user message
- `output`: target empathetic response (MI-adherent, validated, non-directive)

**Key differences from Step 12 (ADP-C training) that Step 14 will apply:**
- `lora_r = 16` (up from 8) -- ADP-A needs more capacity for nuanced empathetic generation
- `max_seq_length = 768` (up from 512) -- multi-turn context windows need more room
- `num_epochs = 3` (down from 20) -- larger dataset converges faster
- Smoke test evaluates warmth, non-directiveness, and absence of clinical language